In [5]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [9]:
ff3 = pd.read_csv("F-F_Research_Data_Factors.csv", skiprows=3)
ff3 = ff3.rename(columns={ff3.columns[0]:"Date"})

ff3 = ff3[ff3["Date"].astype(str).str.isnumeric()]
ff3 = ff3[ff3["Date"].astype(str).str.len()==6]

ff3["Date"] = pd.to_datetime(ff3["Date"].astype(str), format="%Y%m")

ff3[["Mkt-RF","SMB","HML","RF"]] = ff3[["Mkt-RF","SMB","HML","RF"]].astype(float)

ff3.head()

,Date,Mkt-RF,SMB,HML,RF
0,1926-07-01,2.89,-2.55,-2.39,0.22
1,1926-08-01,2.64,-1.14,3.81,0.25
2,1926-09-01,0.38,-1.36,0.05,0.23
3,1926-10-01,-3.27,-0.14,0.82,0.32
4,1926-11-01,2.54,-0.11,-0.61,0.31


In [15]:
cols = ["Mkt-RF", "SMB", "HML"]

mean_list = []
std_list = []
t_list = []

for c in cols:
    x = ff3[c]
    mean_list.append(x.mean())
    std_list.append(x.std())
    t_list.append(x.mean() / (x.std() / np.sqrt(len(x))))

print("Means:")
print(mean_list)
print("Stds:")
print(std_list)
print("T-stats:")
print(t_list)

Means:
[0.6920067170445009, 0.16585222502099084, 0.3456087321578506]
Stds:
[5.313844570644046, 3.1558226444174844, 3.556015725980335]
T-stats:
[4.494251205158074, 1.8136964163778877, 3.3541070129037176]


In [16]:
lag1 = []
lag2 = []
lag12 = []

for c in cols:
    lag1.append(ff3[c].autocorr(lag=1))
    lag2.append(ff3[c].autocorr(lag=2))
    lag12.append(ff3[c].autocorr(lag=12))

print("AC(1):")
print(lag1)
print("AC(2):")
print(lag2)
print("AC(12):")
print(lag12)

AC(1):
[0.09315092762367985, 0.053460990714683805, 0.19558051741354313]
AC(2):
[-0.02337262367606153, 0.06323039361817973, -0.0012903426312245053]
AC(12):
[0.004413896152646186, 0.09763965826933414, 0.04715158289204141]


In [17]:
summary = pd.DataFrame({
    "Mean": mean_list,
    "Std": std_list,
    "T-stat": t_list,
    "AC(1)": lag1,
    "AC(2)": lag2,
    "AC(12)": lag12
}, index=cols)

print("Summary table:")
print(summary)

Summary table:
            Mean       Std    T-stat     AC(1)     AC(2)    AC(12)
Mkt-RF  0.692007  5.313845  4.494251  0.093151 -0.023373  0.004414
SMB     0.165852  3.155823  1.813696  0.053461  0.063230  0.097640
HML     0.345609  3.556016  3.354107  0.195581 -0.001290  0.047152


In [18]:
corr = ff3[cols].corr()
print("Correlation matrix:")
print(corr)

Correlation matrix:
          Mkt-RF       SMB       HML
Mkt-RF  1.000000  0.320081  0.227622
SMB     0.320081  1.000000  0.126010
HML     0.227622  0.126010  1.000000


In [40]:
lines = open("25_Portfolios_5x5.csv", "r").readlines()

start = None
end = None

for i, line in enumerate(lines):
    if start is None and line.strip()[:6].isdigit():
        start = i
    elif start is not None and line.strip() == "":
        end = i
        break

bm = pd.read_csv(
    "25_Portfolios_5x5.csv",
    skiprows=start,
    nrows=end - start - 1
)

bm = bm.rename(columns={bm.columns[0]: "Date"})
bm = bm[bm["Date"].astype(str).str.isnumeric()]
bm["Date"] = pd.to_datetime(bm["Date"], format="%Y%m")

bm = bm.iloc[:, :26]
bm.columns = ["Date"] + [f"ME{i} BM{j}" for i in range(1,6) for j in range(1,6)]

print(bm.head())
print(bm.describe())

        Date  ME1 BM1  ME1 BM2  ME1 BM3  ME1 BM4  ME1 BM5  ME2 BM1  ME2 BM2  \
0 1926-08-01  -2.0206  -8.0282   1.3968   2.1483   8.5104   2.3620  -0.7525   
1 1926-09-01  -4.8291  -2.6806  -4.3417  -3.2683   0.8586  -2.6849  -0.5252   
2 1926-10-01  -9.3633  -3.5519  -3.5024   3.4413  -2.5452  -2.8014  -4.4191   
3 1926-11-01   5.5888   4.1877   2.4384  -4.4495   0.5110   3.1023  -1.7317   
4 1926-12-01  14.9644  -1.4263   0.0084   3.3801   0.8589   3.1000   4.1490   

   ME2 BM3  ME2 BM4  ME2 BM5  ME3 BM1  ME3 BM2  ME3 BM3  ME3 BM4  ME3 BM5  \
0   3.8984   0.2299   6.2937  -0.9386   2.3875   1.9916   4.3179   8.1142   
1   1.0789  -3.2877  -0.9419  -0.3406   0.1689  -1.9453   2.6961  -3.6215   
2  -5.0767  -8.0271  -1.3213  -5.1925  -2.5577   0.4354  -2.1992  -3.1373   
3   3.0425   4.9538   2.7292   1.9110   4.5488   0.4790   4.9524   3.7374   
4   0.8497   2.3043   5.2086  -1.3795   2.3967  -1.0803   3.0189   1.5349   

   ME4 BM1  ME4 BM2  ME4 BM3  ME4 BM4  ME4 BM5  ME5 BM1  ME5 B

In [41]:
merged = bm.merge(ff3, on="Date", how="inner")

print("bm shape:", bm.shape)
print("ff3 shape:", ff3.shape)
print("merged shape:", merged.shape)
print(merged.head())

bm shape: (1190, 26)
ff3 shape: (1191, 5)
merged shape: (1190, 30)
        Date  ME1 BM1  ME1 BM2  ME1 BM3  ME1 BM4  ME1 BM5  ME2 BM1  ME2 BM2  \
0 1926-08-01  -2.0206  -8.0282   1.3968   2.1483   8.5104   2.3620  -0.7525   
1 1926-09-01  -4.8291  -2.6806  -4.3417  -3.2683   0.8586  -2.6849  -0.5252   
2 1926-10-01  -9.3633  -3.5519  -3.5024   3.4413  -2.5452  -2.8014  -4.4191   
3 1926-11-01   5.5888   4.1877   2.4384  -4.4495   0.5110   3.1023  -1.7317   
4 1926-12-01  14.9644  -1.4263   0.0084   3.3801   0.8589   3.1000   4.1490   

   ME2 BM3  ME2 BM4  ME2 BM5  ME3 BM1  ME3 BM2  ME3 BM3  ME3 BM4  ME3 BM5  \
0   3.8984   0.2299   6.2937  -0.9386   2.3875   1.9916   4.3179   8.1142   
1   1.0789  -3.2877  -0.9419  -0.3406   0.1689  -1.9453   2.6961  -3.6215   
2  -5.0767  -8.0271  -1.3213  -5.1925  -2.5577   0.4354  -2.1992  -3.1373   
3   3.0425   4.9538   2.7292   1.9110   4.5488   0.4790   4.9524   3.7374   
4   0.8497   2.3043   5.2086  -1.3795   2.3967  -1.0803   3.0189   1.5349

In [42]:
bm_cols = bm.columns[1:]
stats_index = ["b_MKT","b_SMB","b_HML",
               "t_MKT","t_SMB","t_HML",
               "R2","s(e)",
               "chi2","pvalue"]

results = pd.DataFrame(index=stats_index, columns=bm_cols)
print("Empty results table:")
print(results)

Empty results table:
       ME1 BM1 ME1 BM2 ME1 BM3 ME1 BM4 ME1 BM5 ME2 BM1 ME2 BM2 ME2 BM3  \
b_MKT      NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
b_SMB      NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
b_HML      NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
t_MKT      NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
t_SMB      NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
t_HML      NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
R2         NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
s(e)       NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
chi2       NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
pvalue     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   

       ME2 BM4 ME2 BM5 ME3 BM1 ME3 BM2 ME3 BM3 ME3 BM4 ME3 BM5 ME4 BM1  \
b_MKT      NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
b_SMB      NaN  

In [43]:
print("merged shape:", merged.shape)
print("merged cols contains factors?", ["Mkt-RF" in merged.columns, "SMB" in merged.columns, "HML" in merged.columns])
print(merged[["Date","Mkt-RF","SMB","HML"]].head())

for c in merged.columns:
    if c != "Date":
        merged[c] = pd.to_numeric(merged[c], errors="coerce")

print("dtypes check:")
print(merged[["Mkt-RF","SMB","HML"]].dtypes)

port_cols = bm.columns[1:].tolist()
factor_cols = ["Mkt-RF","SMB","HML"]

reg_df = merged[["Date"] + port_cols + factor_cols].dropna()

print("reg_df shape:", reg_df.shape)
print(reg_df[["Date","Mkt-RF","SMB","HML"]].head())
print("Any NA left?", reg_df.isna().sum().sum())

for c in port_cols:
    y = reg_df[c].values
    X = reg_df[factor_cols]
    X = sm.add_constant(X)

    model = sm.OLS(y, X).fit()
    robust = model.get_robustcov_results(cov_type="HC0")

    w = model.wald_test("SMB=0, HML=0", use_f=False)

    results[c] = [
        robust.params[1], robust.params[2], robust.params[3],
        robust.tvalues[1], robust.tvalues[2], robust.tvalues[3],
        robust.rsquared_adj,
        np.sqrt(robust.mse_resid),
        w.statistic.item(),
        w.pvalue.item()
    ]

print("Filled results preview:")
print(results.iloc[:, :5])

results = results.round(3)
print("Final results (first 6 cols):")
print(results.iloc[:, :6])

merged shape: (1190, 30)
merged cols contains factors? [True, True, True]
        Date  Mkt-RF   SMB   HML
0 1926-08-01    2.64 -1.14  3.81
1 1926-09-01    0.38 -1.36  0.05
2 1926-10-01   -3.27 -0.14  0.82
3 1926-11-01    2.54 -0.11 -0.61
4 1926-12-01    2.62 -0.07  0.06
dtypes check:
Mkt-RF    float64
SMB       float64
HML       float64
dtype: object
reg_df shape: (1190, 29)
        Date  Mkt-RF   SMB   HML
0 1926-08-01    2.64 -1.14  3.81
1 1926-09-01    0.38 -1.36  0.05
2 1926-10-01   -3.27 -0.14  0.82
3 1926-11-01    2.54 -0.11 -0.61
4 1926-12-01    2.62 -0.07  0.06
Any NA left? 0
Filled results preview:
              ME1 BM1      ME1 BM2      ME1 BM3      ME1 BM4      ME1 BM5
b_MKT    1.264389e+00     1.068055     1.045768     0.939842     0.976837
b_SMB    1.474128e+00     1.526765     1.235434     1.229763     1.284733
b_HML    3.559203e-01     0.207581     0.497121     0.577193     0.892318
t_MKT    1.114517e+01    26.626145    29.180854    44.532853    44.514468
t_SMB    8.613

In [44]:
print("bm rows:", bm.shape[0])
print("ff3 rows:", ff3.shape[0])

print("bm Date unique?", bm["Date"].is_unique)
print("ff3 Date unique?", ff3["Date"].is_unique)

print("bm duplicated dates:", bm["Date"].duplicated().sum())
print("ff3 duplicated dates:", ff3["Date"].duplicated().sum())

bm rows: 1190
ff3 rows: 1191
bm Date unique? True
ff3 Date unique? True
bm duplicated dates: 0
ff3 duplicated dates: 0


In [45]:
print(bm.loc[bm["Date"].duplicated(), "Date"].head(10))
print(ff3.loc[ff3["Date"].duplicated(), "Date"].head(10))

Series([], Name: Date, dtype: datetime64[ns])
Series([], Name: Date, dtype: datetime64[ns])


In [46]:
bm = bm.sort_values("Date").drop_duplicates(subset="Date", keep="first")
ff3 = ff3.sort_values("Date").drop_duplicates(subset="Date", keep="first")

print("bm unique now?", bm["Date"].is_unique, bm.shape)
print("ff3 unique now?", ff3["Date"].is_unique, ff3.shape)

bm unique now? True (1190, 26)
ff3 unique now? True (1191, 5)


In [47]:
merged = bm.merge(ff3, on="Date", how="inner")
print("merged shape:", merged.shape)
print(merged.head())
print("merged Date unique?", merged["Date"].is_unique)

merged shape: (1190, 30)
        Date  ME1 BM1  ME1 BM2  ME1 BM3  ME1 BM4  ME1 BM5  ME2 BM1  ME2 BM2  \
0 1926-08-01  -2.0206  -8.0282   1.3968   2.1483   8.5104   2.3620  -0.7525   
1 1926-09-01  -4.8291  -2.6806  -4.3417  -3.2683   0.8586  -2.6849  -0.5252   
2 1926-10-01  -9.3633  -3.5519  -3.5024   3.4413  -2.5452  -2.8014  -4.4191   
3 1926-11-01   5.5888   4.1877   2.4384  -4.4495   0.5110   3.1023  -1.7317   
4 1926-12-01  14.9644  -1.4263   0.0084   3.3801   0.8589   3.1000   4.1490   

   ME2 BM3  ME2 BM4  ME2 BM5  ME3 BM1  ME3 BM2  ME3 BM3  ME3 BM4  ME3 BM5  \
0   3.8984   0.2299   6.2937  -0.9386   2.3875   1.9916   4.3179   8.1142   
1   1.0789  -3.2877  -0.9419  -0.3406   0.1689  -1.9453   2.6961  -3.6215   
2  -5.0767  -8.0271  -1.3213  -5.1925  -2.5577   0.4354  -2.1992  -3.1373   
3   3.0425   4.9538   2.7292   1.9110   4.5488   0.4790   4.9524   3.7374   
4   0.8497   2.3043   5.2086  -1.3795   2.3967  -1.0803   3.0189   1.5349   

   ME4 BM1  ME4 BM2  ME4 BM3  ME4 BM4

In [48]:
for c in merged.columns:
    if c != "Date":
        merged[c] = pd.to_numeric(merged[c], errors="coerce")

port_cols = bm.columns[1:].tolist()
factor_cols = ["Mkt-RF", "SMB", "HML"]

reg_df = merged[["Date"] + port_cols + factor_cols + ["RF"]].dropna()

print("reg_df shape:", reg_df.shape)
print("Any NA left?", reg_df.isna().sum().sum())

reg_df shape: (1190, 30)
Any NA left? 0


In [49]:
stats_index = [
    "b_MKT","b_SMB","b_HML",
    "t_MKT","t_SMB","t_HML",
    "R2","s(e)",
    "chi2","pvalue"
]

results = pd.DataFrame(index=stats_index, columns=port_cols)

print("Empty results table:")
print(results.iloc[:5, :5])

Empty results table:
      ME1 BM1 ME1 BM2 ME1 BM3 ME1 BM4 ME1 BM5
b_MKT     NaN     NaN     NaN     NaN     NaN
b_SMB     NaN     NaN     NaN     NaN     NaN
b_HML     NaN     NaN     NaN     NaN     NaN
t_MKT     NaN     NaN     NaN     NaN     NaN
t_SMB     NaN     NaN     NaN     NaN     NaN


In [51]:
for c in port_cols:
    y = reg_df[c].values
    X = sm.add_constant(reg_df[factor_cols])

    model = sm.OLS(y, X).fit()
    robust = model.get_robustcov_results(cov_type="HC0")

    w = model.wald_test("SMB=0, HML=0", use_f=False)

    results[c] = [
        robust.params[1], robust.params[2], robust.params[3],
        robust.tvalues[1], robust.tvalues[2], robust.tvalues[3],
        robust.rsquared_adj,
        np.sqrt(robust.mse_resid),
        w.statistic.item(),
        w.pvalue.item()
    ]

results = results.round(3)

print(results.iloc[:, :6])

        ME1 BM1   ME1 BM2   ME1 BM3   ME1 BM4   ME1 BM5   ME2 BM1
b_MKT     1.264     1.068     1.046     0.940     0.977     1.083
b_SMB     1.474     1.527     1.235     1.230     1.285     1.149
b_HML     0.356     0.208     0.497     0.577     0.892    -0.226
t_MKT    11.145    26.626    29.181    44.533    44.514    39.941
t_SMB     8.613    10.618    26.553    13.522    21.907    19.949
t_HML     1.402     2.220    10.453    12.718    16.623    -4.457
R2        0.662     0.820     0.888     0.928     0.929     0.909
s(e)      6.923     4.087     2.956     2.219     2.441     2.402
chi2    534.935  1546.680  2357.703  4428.913  5118.071  2493.494
pvalue    0.000     0.000     0.000     0.000     0.000     0.000


In [115]:
import re

print("results shape:", results.shape)

results_rounded = results.copy()

for c in results_rounded.columns:
    if "pvalue" in results_rounded.index:
        p = results_rounded.loc["pvalue", c]
        if p < 0.001:
            results_rounded.loc["pvalue", c] = "<0.001"
        else:
            results_rounded.loc["pvalue", c] = round(p, 3)

results_rounded.loc[results_rounded.index != "pvalue"] = \
    results_rounded.loc[results_rounded.index != "pvalue"].astype(float).round(3)

print(results_rounded.iloc[:, :6])

def _parse_me_bm(name):
    m_me = re.search(r"ME(\d)", name)
    m_bm = re.search(r"BM(\d)", name)
    if (m_me is None) or (m_bm is None):
        return None, None
    return int(m_me.group(1)), int(m_bm.group(1))

def pivot_one_stat(results_df, stat_name):
    s = results_df.loc[stat_name]
    tmp = s.reset_index()
    tmp.columns = ["portfolio", "value"]

    tmp["ME"], tmp["BM"] = zip(*tmp["portfolio"].map(_parse_me_bm))

    bad = tmp[tmp["ME"].isna() | tmp["BM"].isna()]
    if len(bad) > 0:
        print(f"[WARN] {stat_name}: some columns could not be parsed:")
        print(bad.head())

    tmp = tmp.dropna(subset=["ME", "BM"]).copy()
    tmp["ME"] = tmp["ME"].astype(int)
    tmp["BM"] = tmp["BM"].astype(int)

    mat = tmp.pivot(index="ME", columns="BM", values="value").sort_index().sort_index(axis=1)
    mat.index = [f"ME{i}" for i in mat.index]
    mat.columns = [f"BM{j}" for j in mat.columns]
    return mat

print("\nStep: build 5x5 tables for ALL stats")
pivot_tables = {}
for stat in results.index:
    mat = pivot_one_stat(results, stat)
    pivot_tables[stat] = mat
    print(f"\n[{stat}] pivoted shape: {mat.shape}")
    print(mat)

results shape: (10, 25)
        ME1 BM1  ME1 BM2   ME1 BM3   ME1 BM4   ME1 BM5   ME2 BM1
b_MKT     1.264    1.068     1.046      0.94     0.977     1.083
b_SMB     1.474    1.527     1.235      1.23     1.285     1.149
b_HML     0.356    0.208     0.497     0.577     0.892    -0.226
t_MKT    11.145   26.626    29.181    44.533    44.514    39.941
t_SMB     8.613   10.618    26.553    13.522    21.907    19.949
t_HML     1.402     2.22    10.453    12.718    16.623    -4.457
R2        0.662     0.82     0.888     0.928     0.929     0.909
s(e)      6.923    4.087     2.956     2.219     2.441     2.402
chi2    534.935  1546.68  2357.703  4428.913  5118.071  2493.494
pvalue   <0.001   <0.001    <0.001    <0.001    <0.001    <0.001

Step: build 5x5 tables for ALL stats

[b_MKT] pivoted shape: (5, 5)
       BM1    BM2    BM3    BM4    BM5
ME1  1.264  1.068  1.046  0.940  0.977
ME2  1.083  1.019  0.987  0.966  1.063
ME3  1.121  1.015  0.982  0.991  1.110
ME4  1.071  1.026  1.007  1.032  1.1

In [120]:
op = pd.read_csv("25_Portfolios_ME_OP_5x5.csv", skiprows=22)
op = op.rename(columns={op.columns[0]: "Date"})

op = op[op["Date"].astype(str).str.isnumeric()]
op = op[op["Date"].astype(str).str.len() == 6]

op["Date"] = pd.to_datetime(op["Date"].astype(str), format="%Y%m")

op.iloc[:, 1:] = op.iloc[:, 1:].astype(float)

In [121]:
inv = pd.read_csv("25_Portfolios_ME_INV_5x5.csv", skiprows=16)
inv = inv.rename(columns={inv.columns[0]: "Date"})

inv = inv[inv["Date"].astype(str).str.isnumeric()]
inv = inv[inv["Date"].astype(str).str.len() == 6]

inv["Date"] = pd.to_datetime(inv["Date"].astype(str), format="%Y%m")

inv.iloc[:, 1:] = inv.iloc[:, 1:].astype(float)

In [169]:
import pandas as pd

def read_25_portfolios_dynamic(path, ncols=26, label="BM"):
    lines = open(path, "r").readlines()
    start = None
    end = None
    for i, line in enumerate(lines):
        s = line.strip()
        if start is None and (len(s) >= 6) and s[:6].isdigit():
            start = i
        elif start is not None and s == "":
            end = i
            break
    if start is None:
        raise ValueError(f"Could not find data start in {path}")
    if end is None:
        end = len(lines)

    df = pd.read_csv(path, skiprows=start, nrows=end - start - 1)
    df = df.rename(columns={df.columns[0]: "Date"})
    df = df[df["Date"].astype(str).str.isnumeric()]
    df["Date"] = pd.to_datetime(df["Date"].astype(str), format="%Y%m")

    df = df.iloc[:, :ncols]
    df.columns = ["Date"] + [f"ME{i} {label}{j}" for i in range(1, 6) for j in range(1, 6)]
    df.iloc[:, 1:] = df.iloc[:, 1:].astype(float)
    return df

op  = read_25_portfolios_dynamic("25_Portfolios_ME_OP_5x5.csv",  label="OP")
inv = read_25_portfolios_dynamic("25_Portfolios_ME_INV_5x5.csv", label="INV")

ff3 = pd.read_csv("F-F_Research_Data_Factors.csv", skiprows=3)
ff3 = ff3.rename(columns={ff3.columns[0]: "Date"})
ff3 = ff3[ff3["Date"].astype(str).str.isnumeric()]
ff3 = ff3[ff3["Date"].astype(str).str.len() == 6]
ff3["Date"] = pd.to_datetime(ff3["Date"].astype(str), format="%Y%m")
ff3[["Mkt-RF","SMB","HML","RF"]] = ff3[["Mkt-RF","SMB","HML","RF"]].astype(float)

print(op.head())
print(inv.head())
print(ff3.head())

        Date  ME1 OP1  ME1 OP2  ME1 OP3  ME1 OP4  ME1 OP5  ME2 OP1  ME2 OP2  ME2 OP3  ME2 OP4  ME2 OP5  ME3 OP1  ME3 OP2  ME3 OP3  ME3 OP4  ME3 OP5  ME4 OP1  ME4 OP2  ME4 OP3  ME4 OP4  ME4 OP5  \
0 1963-08-01   2.2692   4.0724   2.8528   2.0117   4.2513   6.6139   6.0332   4.1980   4.3652   4.1602   6.4947   4.4979   3.8713   4.5735   7.1690   5.4407   5.3157   6.5548   4.4932   6.7475   
1 1963-09-01  -1.2115  -0.8760  -2.3407   7.4614  -1.4111  -1.9101  -1.8399   0.3819  -2.1290  -4.1133  -0.6188  -1.5697  -1.7206  -1.3673  -3.5884  -1.7840  -0.8339  -2.5080  -2.6017  -2.3103   
2 1963-10-01   0.3526  -0.8650   0.8505   1.3877   0.3652   3.5288   3.2906   1.6911  -1.3657   4.8014   1.7919   2.4473   3.4886   1.5877   1.0544   3.0513   0.3930   1.8017  -0.5941   3.1224   
3 1963-11-01  -2.1603  -1.4852  -2.8550  -3.4152  -2.9596  -1.2879  -0.1063  -0.6418  -0.4184  -1.9823  -1.0905  -0.3619  -2.3565  -3.0870  -2.4164  -0.0982  -1.2203   0.2046  -0.4642   0.4772   
4 1963-12-01  -1.063

In [170]:
import re
import numpy as np
import statsmodels.api as sm

def to_decimal_if_percent(df, cols):
    x = df[cols].astype(float).to_numpy()
    med = np.nanmedian(np.abs(x))
    if med > 1:
        df.loc[:, cols] = df[cols].astype(float) / 100.0
    else:
        df.loc[:, cols] = df[cols].astype(float)
    return df

op  = to_decimal_if_percent(op,  op.columns[1:])
inv = to_decimal_if_percent(inv, inv.columns[1:])
ff3 = to_decimal_if_percent(ff3, ["Mkt-RF","SMB","HML","RF"])

def prep_excess(port_df, ff3_df):
    df = port_df.merge(ff3_df[["Date","Mkt-RF","SMB","HML","RF"]], on="Date", how="inner").sort_values("Date")
    port_cols = [c for c in port_df.columns if c != "Date"]
    for c in port_cols:
        df[c] = df[c] - df["RF"]
    return df, port_cols

def ff3_results(port_df, ff3_df):
    df, port_cols = prep_excess(port_df, ff3_df)
    X = sm.add_constant(df[["Mkt-RF","SMB","HML"]].astype(float))
    stats = ["b_MKT","b_SMB","b_HML","t_MKT","t_SMB","t_HML","R2","s(e)","chi2","pvalue"]
    results = pd.DataFrame(index=stats, columns=port_cols, dtype=object)

    for c in port_cols:
        y = df[c].astype(float)
        m = sm.OLS(y, X, missing="drop").fit()

        results.loc["b_MKT", c] = m.params["Mkt-RF"]
        results.loc["b_SMB", c] = m.params["SMB"]
        results.loc["b_HML", c] = m.params["HML"]

        results.loc["t_MKT", c] = m.tvalues["Mkt-RF"]
        results.loc["t_SMB", c] = m.tvalues["SMB"]
        results.loc["t_HML", c] = m.tvalues["HML"]

        results.loc["R2", c] = m.rsquared
        results.loc["s(e)", c] = np.sqrt(m.mse_resid)

        R = np.array([[0,0,1,0],[0,0,0,1]], dtype=float)
        w = m.wald_test(R)
        results.loc["chi2", c] = float(np.asarray(w.statistic).ravel()[0])
        results.loc["pvalue", c] = float(np.asarray(w.pvalue).ravel()[0])

    return results

def pivot_tables(results_df, label):
    def _parse(name):
        m_me = re.search(r"ME(\d)", name)
        m_q  = re.search(rf"{label}(\d)", name)
        if (m_me is None) or (m_q is None):
            return None, None
        return int(m_me.group(1)), int(m_q.group(1))

    out = {}
    for stat in results_df.index:
        s = results_df.loc[stat]
        tmp = s.reset_index()
        tmp.columns = ["portfolio","value"]
        tmp["ME"], tmp[label] = zip(*tmp["portfolio"].map(_parse))
        tmp = tmp.dropna(subset=["ME", label]).copy()
        tmp["ME"] = tmp["ME"].astype(int)
        tmp[label] = tmp[label].astype(int)

        mat = tmp.pivot(index="ME", columns=label, values="value").sort_index().sort_index(axis=1)
        mat.index = [f"ME{i}" for i in mat.index]
        mat.columns = [f"{label}{j}" for j in mat.columns]
        out[stat] = mat

    return out

results_op  = ff3_results(op, ff3)
results_inv = ff3_results(inv, ff3)

tables_op  = pivot_tables(results_op,  "OP")
tables_inv = pivot_tables(results_inv, "INV")

print("scale check (median abs):",
      np.nanmedian(np.abs(op.iloc[:,1:].to_numpy())),
      np.nanmedian(np.abs(ff3[["Mkt-RF","SMB","HML","RF"]].to_numpy())))

for k in ["b_MKT","b_SMB","b_HML","t_MKT","t_SMB","t_HML","R2","s(e)","chi2","pvalue"]:
    print("\n=== OP:", k, "===\n", tables_op[k])
    print("\n=== INV:", k, "===\n", tables_inv[k])

scale check (median abs): 0.034505 0.01225

=== OP: b_MKT ===
           OP1       OP2       OP3       OP4       OP5
ME1  1.004261  0.910548  0.915175  0.945418  1.098901
ME2  1.120086  1.010197  0.941882  1.003744  1.119898
ME3  1.113119  0.977296  0.974463  1.038771  1.108646
ME4  1.122561  1.031063  1.003905  1.047202  1.052912
ME5  1.126772  1.025024  1.001339     0.986  0.951551

=== INV: b_MKT ===
          INV1      INV2      INV3      INV4      INV5
ME1  1.000388  0.902897  0.896086  0.923084  1.059537
ME2  1.086441  0.962665  0.919437  1.012063  1.143272
ME3  1.072945  0.945645  0.951716  1.018187  1.152606
ME4  1.104885  1.022375  0.987497  1.011509  1.142377
ME5  0.975476  0.902928  0.950341  0.989866  1.130272

=== OP: b_SMB ===
           OP1       OP2       OP3       OP4       OP5
ME1  1.409921  1.035969  0.922067  1.023243  1.086114
ME2  1.089284  0.804058  0.764926  0.785068  0.822778
ME3  0.804216  0.557368  0.497296  0.482007  0.568673
ME4  0.492054  0.318717  0.20937

In [174]:
import numpy as np
import pandas as pd
import re

def returns_summary(port_df, label):
    cols = [c for c in port_df.columns if c != "Date"]
    x = port_df[cols].astype(float)

    summ = pd.DataFrame({
        "mean": x.mean(),
        "std": x.std(ddof=1),
        "sharpe": x.mean() / x.std(ddof=1),
        "min": x.min(),
        "max": x.max()
    })

    def _parse(name):
        m_me = re.search(r"ME(\d)", name)
        m_q = re.search(rf"{label}(\d)", name)
        if (m_me is None) or (m_q is None):
            return None, None
        return int(m_me.group(1)), int(m_q.group(1))

    tmp = summ.reset_index().rename(columns={"index":"portfolio"})
    tmp["ME"], tmp[label] = zip(*tmp["portfolio"].map(_parse))
    tmp = tmp.dropna(subset=["ME", label]).copy()
    tmp["ME"] = tmp["ME"].astype(int)
    tmp[label] = tmp[label].astype(int)

    mean_mat = tmp.pivot(index="ME", columns=label, values="mean").sort_index().sort_index(axis=1)
    sd_mat   = tmp.pivot(index="ME", columns=label, values="std").sort_index().sort_index(axis=1)
    sh_mat   = tmp.pivot(index="ME", columns=label, values="sharpe").sort_index().sort_index(axis=1)

    mean_mat.index = [f"ME{i}" for i in mean_mat.index]
    sd_mat.index   = [f"ME{i}" for i in sd_mat.index]
    sh_mat.index   = [f"ME{i}" for i in sh_mat.index]

    mean_mat.columns = [f"{label}{j}" for j in mean_mat.columns]
    sd_mat.columns   = [f"{label}{j}" for j in sd_mat.columns]
    sh_mat.columns   = [f"{label}{j}" for j in sh_mat.columns]

    return summ, mean_mat, sd_mat, sh_mat

summ_op, mean_op, sd_op, sharpe_op = returns_summary(op, "OP")
summ_inv, mean_inv, sd_inv, sharpe_inv = returns_summary(inv, "INV")

print("\n=== OP: return summary (top 10 by mean) ===")
print(summ_op.sort_values("mean", ascending=False).head(10))

print("\n=== INV: return summary (top 10 by mean) ===")
print(summ_inv.sort_values("mean", ascending=False).head(10))

print("\n=== OP: mean (5x5) ===\n", mean_op)
print("\n=== OP: std  (5x5) ===\n", sd_op)
print("\n=== OP: sharpe (5x5) ===\n", sharpe_op)

print("\n=== INV: mean (5x5) ===\n", mean_inv)
print("\n=== INV: std  (5x5) ===\n", sd_inv)
print("\n=== INV: sharpe (5x5) ===\n", sharpe_inv)


=== OP: return summary (top 10 by mean) ===
             mean       std    sharpe       min       max
ME1 OP4  0.013754  0.069082  0.199099 -0.292118  0.896401
ME2 OP5  0.013041  0.064231  0.203031 -0.328563  0.268020
ME1 OP2  0.012896  0.058898  0.218949 -0.283566  0.294406
ME3 OP5  0.012455  0.059483  0.209390 -0.277793  0.243127
ME1 OP3  0.012364  0.056902  0.217282 -0.288345  0.267149
ME2 OP3  0.011868  0.054071  0.219496 -0.271331  0.250255
ME4 OP5  0.011748  0.053391  0.220037 -0.251660  0.213431
ME1 OP5  0.011621  0.068439  0.169807 -0.326825  0.271517
ME2 OP4  0.011577  0.057589  0.201024 -0.270295  0.280189
ME3 OP4  0.011549  0.053932  0.214141 -0.246470  0.224992

=== INV: return summary (top 10 by mean) ===
              mean       std    sharpe       min       max
ME1 INV3  0.013401  0.057235  0.234139 -0.284616  0.282750
ME1 INV1  0.013267  0.074141  0.178940 -0.327047  0.442390
ME1 INV2  0.012911  0.057906  0.222956 -0.276557  0.267683
ME3 INV1  0.012642  0.058485  0.216

In [175]:
import numpy as np
import pandas as pd

def model_fit_summary(results_df, name):
    r2 = results_df.loc["R2"].astype(float)
    se = results_df.loc["s(e)"].astype(float)
    chi2 = results_df.loc["chi2"].astype(float)
    pv = results_df.loc["pvalue"].astype(float)

    out = pd.Series({
        "N": r2.shape[0],
        "R2_mean": r2.mean(),
        "R2_median": r2.median(),
        "R2_min": r2.min(),
        "R2_max": r2.max(),
        "s(e)_mean": se.mean(),
        "s(e)_median": se.median(),
        "p<0.05_share": (pv < 0.05).mean(),
        "p<0.01_share": (pv < 0.01).mean(),
        "chi2_mean": chi2.mean(),
        "chi2_median": chi2.median()
    }, name=name)

    return out

fit_op = model_fit_summary(results_op, "OP")
fit_inv = model_fit_summary(results_inv, "INV")

print("\n=== FF3 fit summary ===")
print(pd.concat([fit_op, fit_inv], axis=1))


=== FF3 fit summary ===
                      OP         INV
N              25.000000   25.000000
R2_mean         0.893587    0.914570
R2_median       0.903833    0.922056
R2_min          0.697683    0.796638
R2_max          0.939889    0.954665
s(e)_mean       0.018321    0.015965
s(e)_median     0.016770    0.015174
p<0.05_share    0.960000    1.000000
p<0.01_share    0.960000    1.000000
chi2_mean     497.506405  723.610657
chi2_median   334.313194  485.104390
